[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/streaming-interruption.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239464-lesson-1-streaming)

# 流式处理

## 回顾

在模块2中，我们介绍了几种自定义图状态和内存的方法。

我们构建了一个具有外部内存的聊天机器人，可以维持长时间的对话。

## 目标

本模块将深入探讨`人工参与循环`，这建立在内存的基础上，允许用户以各种方式直接与图进行交互。

为了为`人工参与循环`奠定基础，我们首先将深入研究流式处理，它提供了几种在执行过程中可视化图输出（例如，节点状态或聊天模型tokens）的方法。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_openai langgraph_sdk

## 流式处理

LangGraph内置了[对流式处理的一流支持](https://langchain-ai.github.io/langgraph/concepts/low_level/#streaming)。

让我们设置模块2中的聊天机器人，并展示在执行期间从图中流式输出的各种方法。

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

注意我们在`call_model`中使用`RunnableConfig`来启用按token的流式处理。这[仅在python < 3.11时需要](https://langchain-ai.github.io/langgraph/how-tos/streaming-tokens/)。我们包含此项以防您在CoLab中运行此notebook，它将使用python 3.x。

In [ ]:
from IPython.display import Image, display

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage
from langchain_core.runnables import RunnableConfig

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState

# LLM
model = ChatOpenAI(model="gpt-4o", temperature=0) 

# 状态
class State(MessagesState):
    summary: str

# 定义调用模型的逻辑
def call_model(state: State, config: RunnableConfig):
    """调用LLM并返回响应"""
    
    # 获取摘要（如果存在）
    summary = state.get("summary", "")

    # 如果有摘要，则我们添加它
    if summary:
        
        # 将摘要添加到系统消息
        system_message = f"之前的对话摘要：{summary}"

        # 将摘要附加到任何更新的消息
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages, config)
    return {"messages": response}

def summarize_conversation(state: State):
    """总结对话内容"""
    
    # 首先，我们获取任何现有的摘要
    summary = state.get("summary", "")

    # 创建我们的总结提示
    if summary:
        
        # 摘要已经存在
        summary_message = (
            f"这是迄今为止对话的摘要：{summary}\n\n"
            "通过考虑上面的新消息来扩展摘要："
        )
        
    else:
        summary_message = "创建上述对话的摘要："

    # 将提示添加到我们的历史记录
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 删除除最近的2条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

# 确定是结束还是总结对话
def should_continue(state: State):
    """返回要执行的下一个节点。"""
    
    messages = state["messages"]
    
    # 如果有超过六条消息，那么我们总结对话
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否则我们可以直接结束
    return END

# 定义新图
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 设置入口点为conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 编译
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

### 流式处理完整状态

现在，让我们谈论[流式处理我们的图状态](https://langchain-ai.github.io/langgraph/concepts/low_level/#streaming)的方法。

`.stream`和`.astream`是用于流式返回结果的同步和异步方法。

LangGraph支持几种不同的[图状态流式模式](https://langchain-ai.github.io/langgraph/how-tos/stream-values/)：

* `values`：在调用每个节点后流式传输图的完整状态。
* `updates`：在调用每个节点后流式传输图状态的更新。

![values_vs_updates.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbaf892d24625a201744e5_streaming1.png)

让我们看看`stream_mode="updates"`。

因为我们使用`updates`进行流式处理，所以我们只看到图中每个节点运行后状态的更新。

每个`chunk`是一个字典，其中`node_name`作为键，更新的状态作为值。

In [ ]:
# 创建线程
config = {"configurable": {"thread_id": "1"}}

# 开始对话
for chunk in graph.stream({"messages": [HumanMessage(content="你好！我是Lance")]}, config, stream_mode="updates"):
    print(chunk)

现在让我们只打印状态更新。

In [ ]:
# 开始对话
for chunk in graph.stream({"messages": [HumanMessage(content="你好！我是Lance")]}, config, stream_mode="updates"):
    chunk['conversation']["messages"].pretty_print()

现在，我们可以看到`stream_mode="values"`。

这是调用`conversation`节点后图的`完整状态`。

In [ ]:
# 再次开始对话
config = {"configurable": {"thread_id": "2"}}

# 开始对话
input_message = HumanMessage(content="你好！我是Lance")
for event in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    for m in event['messages']:
        m.pretty_print()
    print("---"*25)

### 流式处理tokens

我们通常想要流式处理的不仅仅是图状态。

特别是，对于聊天模型调用，通常会在生成tokens时对其进行流式处理。

我们可以[使用`.astream_events`方法](https://langchain-ai.github.io/langgraph/how-tos/streaming-from-final-node/#stream-outputs-from-the-final-node)来做到这一点，它会在节点内发生事件时将其流式传输回来！

每个事件都是一个有几个键的字典：

* `event`：这是正在发出的事件类型。
* `name`：这是事件的名称。
* `data`：这是与事件相关的数据。
* `metadata`：包含`langgraph_node`，即发出事件的节点。

让我们看一下。

In [ ]:
config = {"configurable": {"thread_id": "3"}}
input_message = HumanMessage(content="告诉我关于49ers NFL球队的信息")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    print(f"节点：{event['metadata'].get('langgraph_node','')}. 类型：{event['event']}. 名称：{event['name']}")

关键点是图中聊天模型的tokens具有`on_chat_model_stream`类型。

我们可以使用`event['metadata']['langgraph_node']`来选择要流式传输的节点。

我们可以使用`event['data']`来获取每个事件的实际数据，在这种情况下是`AIMessageChunk`。

In [ ]:
node_to_stream = 'conversation'
config = {"configurable": {"thread_id": "4"}}
input_message = HumanMessage(content="告诉我关于49ers NFL球队的信息")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    # 从特定节点获取聊天模型tokens
    if event["event"] == "on_chat_model_stream" and event['metadata'].get('langgraph_node','') == node_to_stream:
        print(event["data"])

如上所示，只需使用`chunk`键来获取`AIMessageChunk`。

In [ ]:
config = {"configurable": {"thread_id": "5"}}
input_message = HumanMessage(content="告诉我关于49ers NFL球队的信息")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    # 从特定节点获取聊天模型tokens
    if event["event"] == "on_chat_model_stream" and event['metadata'].get('langgraph_node','') == node_to_stream:
        data = event["data"]
        print(data["chunk"].content, end="|")

### 与LangGraph API一起使用流式处理

**⚠️ 免责声明**

自从拍摄这些视频以来，我们更新了Studio，使其可以在本地运行并在浏览器中打开。这现在是运行Studio的首选方式（而不是像视频中显示的那样使用桌面应用程序）。请参阅[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)的本地开发服务器文档和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)的相关说明。要启动本地开发服务器，请在此模块的`/studio`目录中的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

LangGraph API [支持编辑图状态](https://langchain-ai.github.io/langgraph/cloud/how-tos/human_in_the_loop_edit_state/#initial-invocation)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("很抱歉，Google Colab目前不支持LangGraph Studio")

In [ ]:
from langgraph_sdk import get_client

# 这是本地开发服务器的URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

让我们像之前一样[流式传输`values`](https://langchain-ai.github.io/langgraph/cloud/how-tos/stream_values/)。

In [ ]:
# 创建新线程
thread = await client.threads.create()
# 输入消息
input_message = HumanMessage(content="计算2乘以3")
async for event in client.runs.stream(thread["thread_id"], 
                                      assistant_id="agent", 
                                      input={"messages": [input_message]}, 
                                      stream_mode="values"):
    print(event)

流式对象具有：

* `event`：类型
* `data`：状态

In [ ]:
from langchain_core.messages import convert_to_messages
thread = await client.threads.create()
input_message = HumanMessage(content="计算2乘以3")
async for event in client.runs.stream(thread["thread_id"], assistant_id="agent", input={"messages": [input_message]}, stream_mode="values"):
    messages = event.data.get('messages',None)
    if messages:
        print(convert_to_messages(messages)[-1])
    print('='*25)

有一些只通过API支持的新流式模式。

例如，我们可以[使用`messages`模式](https://langchain-ai.github.io/langgraph/cloud/how-tos/stream_messages/)来更好地处理上述情况！

此模式目前假设您的图中有一个`messages`键，它是一个消息列表。

使用`messages`模式发出的所有事件都有两个属性：

* `event`：这是事件的名称
* `data`：这是与事件相关的数据

In [ ]:
thread = await client.threads.create()
input_message = HumanMessage(content="计算2乘以3")
async for event in client.runs.stream(thread["thread_id"], 
                                      assistant_id="agent", 
                                      input={"messages": [input_message]}, 
                                      stream_mode="messages"):
    print(event.event)

我们可以看到几个事件：

* `metadata`：关于运行的元数据
* `messages/complete`：完整形成的消息
* `messages/partial`：聊天模型tokens

您可以在[这里](https://langchain-ai.github.io/langgraph/cloud/concepts/api/#modemessages)进一步了解这些类型。

现在，让我们展示如何流式传输这些消息。

我们将定义一个辅助函数来更好地格式化消息中的工具调用。

In [ ]:
thread = await client.threads.create()
input_message = HumanMessage(content="计算2乘以3")

def format_tool_calls(tool_calls):
    """
    将工具调用列表格式化为可读字符串。

    Args:
        tool_calls (list): 字典列表，每个表示一个工具调用。
            每个字典应该有'id'、'name'和'args'键。

    Returns:
        str: 格式化的工具调用字符串，如果列表为空则返回"无工具调用"。

    """

    if tool_calls:
        formatted_calls = []
        for call in tool_calls:
            formatted_calls.append(
                f"工具调用ID：{call['id']}，函数：{call['name']}，参数：{call['args']}"
            )
        return "\n".join(formatted_calls)
    return "无工具调用"

async for event in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input={"messages": [input_message]},
    stream_mode="messages",):
    
    # 处理元数据事件
    if event.event == "metadata":
        print(f"元数据：运行ID - {event.data['run_id']}")
        print("-" * 50)
    
    # 处理部分消息事件
    elif event.event == "messages/partial":
        for data_item in event.data:
            # 处理用户消息
            if "role" in data_item and data_item["role"] == "user":
                print(f"人类：{data_item['content']}")
            else:
                # 从事件中提取相关数据
                tool_calls = data_item.get("tool_calls", [])
                invalid_tool_calls = data_item.get("invalid_tool_calls", [])
                content = data_item.get("content", "")
                response_metadata = data_item.get("response_metadata", {})

                if content:
                    print(f"AI：{content}")

                if tool_calls:
                    print("工具调用：")
                    print(format_tool_calls(tool_calls))

                if invalid_tool_calls:
                    print("无效工具调用：")
                    print(format_tool_calls(invalid_tool_calls))

                if response_metadata:
                    finish_reason = response_metadata.get("finish_reason", "N/A")
                    print(f"响应元数据：完成原因 - {finish_reason}")
                    
        print("-" * 50)